In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        _tok = ""
        try:
            from google.colab import userdata
            _tok = userdata.get("GH_TOKEN") or ""
        except Exception:
            _tok = ""
        if not _tok:
            print("WARNING: no 'GH_TOKEN' Colab secret found; cloning this PRIVATE repo will fail.\n"
                  "Add a GitHub token (repo scope) via the key icon (Secrets) as 'GH_TOKEN', then re-run.")
        _url = (f"https://{_tok}@github.com/{_slug}.git" if _tok
                else f"https://github.com/{_slug}.git")
        subprocess.run(["git", "clone", "--depth", "1", _url, str(_root)], check=True)
        subprocess.run(["git", "-C", str(_root), "remote", "set-url", "origin",
                        f"https://github.com/{_slug}.git"])  # keep the token out of the saved remote
    os.chdir(_root / "06-gateway/identity-security/agentic-identity-gcp-lab/notebooks/practice")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


    # 08 · Audit, kill switches and anomaly detection — practice

    **Primer section:** §9. Generate history, query it, flip two kill switches, and write the anomaly
    heuristic.

> **How to use this practice notebook.** Every `____` is a blank you must fill (a name, an
> argument, an expression); a `raise NotImplementedError("fill me")` means "write the body".
> Each exercise ends with `assert` checks — run the cell, and if it is silent you got it right.
> The completed version lives in `notebooks/solutions/`.

In [ ]:
import logging

from agentsec.logging_utils import quiet_logs

quiet_logs(logging.ERROR)

from collections import Counter, defaultdict
from datetime import datetime

from agentsec.agents import REFUNDS, LocalStack, Step, reset_demo_state
from agentsec.audit import AuditLog
from agentsec.config import Settings
from agentsec.runtime import confirm, resume_after_auth, run_turn, seed_session

reset_demo_state()

USER = {"subject": "u-ana", "email": "ana@customer.example", "tenant": "acme"}
SCOPES = ["customers:read", "orders:read", "payments:refund", "email:send"]

audit = AuditLog()
stack = LocalStack.create(Settings(), audit=audit)
AGENT = stack.agent_id.spiffe_id

def short(spiffe: str) -> str:
    return spiffe.rsplit("/", 1)[-1]

## Exercise 1 — generate history

Run one turn for Ana with a read, `run_sql`, a 35 USD refund and a 120 USD refund; approve the
pending confirmation. Then one turn for Ben (scopes: only `customers:read`) that tries a refund.

In [ ]:
await seed_session(stack.runner, user_id="u-ana", session_id="ana-1", user=USER, scopes=SCOPES)
stack.script(
    Step.call("lookup_customer", email="ana@customer.example"),
    Step.call("run_sql", query="select 1"),
    Step.call("issue_refund", order_id="O-5002", amount=35.0, currency="USD", reason="dup"),
    Step.call("issue_refund", order_id="O-5001", amount=120.0, currency="USD", reason="cancelled"),
    Step.say("done"),
)
r = await run_turn(stack.runner, user_id="u-ana", session_id="ana-1", message="refund my orders")
stack.script(Step.say("issued"))
await confirm(stack.runner, user_id="u-ana", session_id="ana-1", pending=____, confirmed=____)

BEN = {"subject": "u-ben", "email": "ben@customer.example", "tenant": "acme"}
await seed_session(stack.runner, user_id="u-ben", session_id="ben-1", user=BEN, scopes=[____])
stack.script(Step.call("issue_refund", order_id="O-5003", amount=20.0, currency="USD", reason="x"), Step.say("no"))
await run_turn(stack.runner, user_id="u-ben", session_id="ben-1", message="refund O-5003")

assert len(REFUNDS) == 2
assert len(audit.events()) > 10
print(len(audit.events()), "events")

## Exercise 2 — answer the investigator's questions

Using only `audit`, compute: the set of denied tools, the approver of the 120 USD refund, and Ben's
denial reason. Every event must carry the agent's SPIFFE ID.

In [ ]:
denied_tools = {e.tool for e in audit.____()}
approver = next(e.____ for e in audit.events() if e.____)
ben_denial = next(e for e in audit.denials() if e.____ == "ben@customer.example")

assert denied_tools == {"run_sql", "issue_refund"}
assert approver == "ana@customer.example"
assert "missing scopes" in ben_denial.reasons[0]
assert all(e.agent == AGENT for e in audit.events())
identities = Counter((e.user, e.authority) for e in audit.events())
assert set(identities) == {("ana@customer.example", "delegated"), ("ben@customer.example", "delegated")}
print("denied:", denied_tools, "| approver:", approver, "| ben:", ben_denial.reasons[0])
print("identities:", dict(identities))

## Exercise 3 — the policy kill switch

Prove a refund works, remove the agent from `issue_refund`'s allow list, prove the next refund is
denied *without* recreating the stack, then restore.

In [ ]:
await seed_session(stack.runner, user_id="u-ana", session_id="ana-2", user=USER, scopes=SCOPES)

async def try_refund():
    stack.script(Step.call("issue_refund", order_id="O-5002", amount=10.0, currency="USD", reason="test"), Step.say("ok"))
    r = await run_turn(stack.runner, user_id="u-ana", session_id="ana-2", message="refund")
    return next(t["response"] for t in r.tool_responses if t["name"] == "issue_refund")

before = await try_refund()
saved = list(stack.engine.policy.tools["issue_refund"].allow)
stack.engine.policy.tools["issue_refund"].____ = ____
after = await try_refund()
stack.engine.policy.tools["issue_refund"].allow = saved
restored = await try_refund()

assert before["status"] == "issued"
assert after["error"] == "policy_denied" and "not in allow list" in after["reasons"][0]
assert restored["status"] == "issued"
print("before:", before["status"], "| after:", after["reasons"][0], "| restored:", restored["status"])

## Exercise 4 — revoke consent

Get CRM consent for Ana through the ADK round trip, revoke it in the broker, and show that the next
`crm_lookup` pauses for consent again.

In [ ]:
await seed_session(stack.runner, user_id="u-ana", session_id="ana-3", user=USER, scopes=SCOPES)
stack.script(Step.call("crm_lookup", email="ana@customer.example"), Step.say("ok"))
r = await run_turn(stack.runner, user_id="u-ana", session_id="ana-3", message="crm")
pending = r.pending_auth[0]
stack.auth_manager.finalize(auth_provider=stack.crm_provider, user_id="u-ana", consent_nonce=pending.consent_nonce)
stack.script(Step.say("ok"))
r2 = await resume_after_auth(stack.runner, user_id="u-ana", session_id="ana-3", pending=pending)
assert "user-delegated token" in next(t["response"] for t in r2.tool_responses if t["name"] == "crm_lookup")["content"]

stack.auth_manager.____(auth_provider=stack.crm_provider, user_id="u-ana")

stack.script(Step.call("crm_lookup", email="ana@customer.example"), Step.say("ok"))
r3 = await run_turn(stack.runner, user_id="u-ana", session_id="ana-3", message="crm again")
assert len(r3.____) == 1
assert [e.outcome for e in stack.auth_manager.access_log] == ["uri_consent_required", "success", "uri_consent_required"]
print("consent required again after revoke")

## Exercise 5 — write the anomaly heuristic

Script a burst of six 1 USD refunds in one turn, then complete `destructive_bursts`: over
`tool.decision` events for destructive tools, flag any agent with more than `threshold` attempts
inside a `window_s`-second window.

In [ ]:
await seed_session(stack.runner, user_id="u-ana", session_id="burst", user=USER, scopes=SCOPES)
stack.script(*[Step.call("issue_refund", order_id="O-5002", amount=1.0, currency="USD", reason=f"loop {i}") for i in range(6)], Step.say("done"))
await run_turn(stack.runner, user_id="u-ana", session_id="burst", message="refund")

DESTRUCTIVE = {name for name, tp in stack.policy.tools.items() if tp.tier.value == "destructive"}

def destructive_bursts(log: AuditLog, *, window_s: int = 60, threshold: int = 5) -> list[tuple[str, str, int]]:
    by_agent: dict[str, list[datetime]] = defaultdict(list)
    for e in log.events(lambda e: e.event_type == "____" and e.tool in DESTRUCTIVE):
        by_agent[short(e.agent)].append(datetime.fromisoformat(e.ts))
    alerts = []
    for agent, times in by_agent.items():
        times.sort()
        for i, start in enumerate(times):
            n = sum(1 for t in times[i:] if (t - start).total_seconds() <= ____)
            if n > ____:
                alerts.append((agent, start.strftime("%H:%M:%S"), n))
                break
    return alerts

alerts = destructive_bursts(audit)
assert alerts and all(agent == "support-agent" for agent, _, _ in alerts)
assert alerts[0][2] >= 6
print("alerts:", alerts)

**In one sentence:** "One event per tool call with both identities, the decision, the reasons
and the approver; kill switches that work on the next call — a deny policy or pulling the allow-list
entry, revoking consent in the broker; and anomaly detection over the denials and destructive attempts."